# Chapter 21: Breadth-First and Depth-First Search

The programs from Chapter 21 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Breadth-First Search

`search.py`

In [1]:
"""Breadth-first and depth-first search, and what they're good for.

Graphs here are plain dicts mapping each vertex to a list of neighbors,
like the adjacency lists of Chapter 20.
"""

from collections import deque


def bfs(graph, start):
    """Fewest edges from start to every reachable vertex, and each
    vertex's parent on such a shortest path."""
    dist, parent = {start: 0}, {start: None}
    queue = deque([start])
    while queue:
        v = queue.popleft()
        for w in graph[v]:
            if w not in dist:  # first visit is by a shortest path
                dist[w] = dist[v] + 1
                parent[w] = v
                queue.append(w)
    return dist, parent


def shortest_path(graph, start, goal):
    """A path with the fewest edges, as a list, or None."""
    _, parent = bfs(graph, start)
    if goal not in parent:
        return None
    path = [goal]
    while parent[path[-1]] is not None:
        path.append(parent[path[-1]])
    return path[::-1]


def dfs(graph, start):
    """Vertices in depth-first order, using an explicit stack."""
    seen, order, stack = set(), [], [start]
    while stack:
        v = stack.pop()
        if v in seen:
            continue
        seen.add(v)
        order.append(v)
        for w in reversed(graph[v]):  # so the first neighbor comes next
            if w not in seen:
                stack.append(w)
    return order


def components(graph):
    """The connected components of an undirected graph."""
    seen, out = set(), []
    for v in graph:
        if v not in seen:
            part = dfs(graph, v)
            seen.update(part)
            out.append(part)
    return out


def topological_order(graph):
    """Order a directed graph so every edge points forward (Kahn's
    algorithm), or return None if there is a cycle."""
    indegree = dict.fromkeys(graph, 0)
    for v in graph:
        for w in graph[v]:
            indegree[w] += 1
    ready = deque(v for v in graph if indegree[v] == 0)
    order = []
    while ready:
        v = ready.popleft()
        order.append(v)
        for w in graph[v]:
            indegree[w] -= 1
            if indegree[w] == 0:
                ready.append(w)
    return order if len(order) == len(graph) else None


def two_colors(graph):
    """Color an undirected graph with two colors so no edge joins two
    of the same color (bipartite), or return None if impossible."""
    color = {}
    for start in graph:
        if start in color:
            continue
        color[start] = 0
        queue = deque([start])
        while queue:
            v = queue.popleft()
            for w in graph[v]:
                if w not in color:
                    color[w] = 1 - color[v]
                    queue.append(w)
                elif color[w] == color[v]:
                    return None
    return color

`route_demo.py`

In [2]:
"""Fewest stops between stations on a small (fictional) metro."""

from search import bfs, dfs, shortest_path

lines = {
    "Red": ["Harbor", "Market", "Central", "Park", "Zoo"],
    "Blue": ["Airport", "Central", "Museum", "Stadium"],
    "Green": ["Market", "Museum", "University"],
}
metro = {}
for stations in lines.values():
    for a, b in zip(stations, stations[1:]):
        metro.setdefault(a, []).append(b)
        metro.setdefault(b, []).append(a)

print(" -> ".join(shortest_path(metro, "Harbor", "Stadium")))
print(" -> ".join(shortest_path(metro, "Zoo", "University")))
dist, _ = bfs(metro, "Central")
for stops in sorted(set(dist.values())):
    names = [s for s, d in dist.items() if d == stops]
    print(f"{stops} away from Central: {', '.join(names)}")
print("depth-first from Harbor:")
print(", ".join(dfs(metro, "Harbor")))

Harbor -> Market -> Museum -> Stadium
Zoo -> Park -> Central -> Museum -> University
0 away from Central: Central
1 away from Central: Market, Park, Airport, Museum
2 away from Central: Harbor, Zoo, Stadium, University
depth-first from Harbor:
Harbor, Market, Central, Park, Zoo, Airport, Museum, Stadium, University


## Ordering Tasks: Topological Sort

`course_order.py`

In [3]:
"""In which order can the courses be taken? An edge A -> B means A must
come before B: a topological order of the prerequisite graph."""

from search import topological_order

before = {
    "Python": ["Data Structures", "Web"],
    "Math": ["Algorithms", "Statistics"],
    "Data Structures": ["Algorithms"],
    "Algorithms": ["Machine Learning"],
    "Statistics": ["Machine Learning"],
    "Web": [],
    "Machine Learning": [],
}
for i, course in enumerate(topological_order(before), 1):
    print(i, course)
before["Machine Learning"].append("Python")  # a cycle: impossible
print(topological_order(before))

1 Python
2 Math
3 Data Structures
4 Web
5 Statistics
6 Algorithms
7 Machine Learning
None


## Timing BFS: When Does the Queue Matter?

`bench_bfs.py`

In [4]:
"""BFS with a deque as the queue, and with a list using pop(0), on two
graphs: a square grid of n cells, and a star (one center vertex joined
to n others)."""

from collections import deque

import bench


def star(n):
    g = {(0, 0): [(i, 1) for i in range(n)]}
    for i in range(n):
        g[i, 1] = [(0, 0)]
    return (g,)


def grid(n):
    side = int(n**0.5)
    g = {}
    for r in range(side):
        for c in range(side):
            g[r, c] = [
                (r + dr, c + dc)
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1))
                if 0 <= r + dr < side and 0 <= c + dc < side
            ]
    return (g,)


def bfs_deque(g):
    seen, queue = {(0, 0)}, deque([(0, 0)])
    while queue:
        v = queue.popleft()
        for w in g[v]:
            if w not in seen:
                seen.add(w)
                queue.append(w)


def bfs_list(g):
    seen, queue = {(0, 0)}, [(0, 0)]
    while queue:
        v = queue.pop(0)
        for w in g[v]:
            if w not in seen:
                seen.add(w)
                queue.append(w)


print("grid of n cells:")
sizes = [10_000, 40_000, 160_000, 640_000]
bench.table(
    {
        "deque": bench.doubling(bfs_deque, grid, sizes),
        "list.pop(0)": bench.doubling(bfs_list, grid, sizes),
    }
)
print("star with n leaves:")
sizes = [10_000, 20_000, 40_000, 80_000]
bench.table(
    {
        "deque": bench.doubling(bfs_deque, star, sizes),
        "list.pop(0)": bench.doubling(bfs_list, star, sizes),
    }
)

grid of n cells:


         n           deque     list.pop(0)
    10,000         2.62 ms         3.08 ms
    40,000    21.8 ms x8.3      25 ms x8.1
   160,000     117 ms x5.4     117 ms x4.7
   640,000     731 ms x6.2     745 ms x6.4
star with n leaves:


         n           deque     list.pop(0)
    10,000          1.5 ms         7.38 ms
    20,000    3.61 ms x2.4    34.1 ms x4.6
    40,000    7.22 ms x2.0     133 ms x3.9
    80,000    20.4 ms x2.8     528 ms x4.0
